In [2]:
import pandas as pd

data_H = pd.read_parquet('data/obs_final_elements_geometry_H.parquet', engine='pyarrow')
rubin = pd.read_parquet('data/obs_sbn_X05_full_mpc_desig.parquet', engine='pyarrow')

In [3]:
data_H.columns

Index(['obstime', 'mpc_row', 'mpc_designation', 'permid', 'provid', 'obs_et',
       'dt_from_epoch_days', 'a', 'e', 'q', 'incl', 'node', 'peri', 'M', 'r',
       'delta', 'phase_angle', 'mag', 'H'],
      dtype='object')

In [4]:
rubin.columns

Index(['id', 'trksub', 'trkid', 'obsid', 'submission_id',
       'submission_block_id', 'obs80', 'status', 'ref', 'healpix', 'permid',
       'provid', 'artsat', 'mode', 'stn', 'trx', 'rcv', 'sys', 'ctr', 'pos1',
       'pos2', 'pos3', 'poscov11', 'poscov12', 'poscov13', 'poscov22',
       'poscov23', 'poscov33', 'prog', 'obstime', 'ra', 'dec', 'rastar',
       'decstar', 'obscenter', 'deltara', 'deltadec', 'dist', 'pa', 'rmsra',
       'rmsdec', 'rmsdist', 'rmspa', 'rmscorr', 'delay', 'rmsdelay', 'doppler',
       'rmsdoppler', 'astcat', 'mag', 'rmsmag', 'band', 'photcat', 'photap',
       'nucmag', 'logsnr', 'seeing', 'exp', 'rmsfit', 'com', 'frq', 'disc',
       'subfrm', 'subfmt', 'prectime', 'precra', 'precdec', 'unctime', 'notes',
       'remarks', 'deprecated', 'localuse', 'nstars', 'prev_desig', 'prev_ref',
       'rmstime', 'created_at', 'updated_at', 'trkmpc', 'orbit_id',
       'designation_asterisk', 'shapeocc', 'obssubid', 'replacesobsid',
       'group_id', 'vel1', 'fltr'

In [10]:
obs_magnitudes = rubin[['mpc_designation', 'obstime', 'band', 'mag', 'rmsmag']].dropna(subset=['mpc_designation']).copy()
print(f"Kept {len(obs_magnitudes):,} of {len(rubin):,} rows")
obs_magnitudes.head(10)

Kept 7,954,397 of 8,078,546 rows


,mpc_designation,obstime,band,mag,rmsmag
552,2025 MD8,2025-05-02 00:42:01.796,i,23.096,0.124
553,2025 MD8,2025-05-02 00:39:54.996,i,22.927,0.127
554,2025 MD8,2025-05-02 01:01:01.387,i,23.055,0.105
555,2025 MD8,2025-05-02 01:03:10.218,i,23.059,0.129
556,2025 MD8,2025-05-02 01:29:18.645,r,22.911,0.072
557,2025 MD8,2025-05-02 02:46:01.182,g,23.957,0.171
558,2025 MP24,2025-05-02 02:41:08.571,g,24.027,0.152
559,2025 MP24,2025-05-02 01:50:19.014,r,22.737,0.065
560,2025 MP24,2025-05-02 01:52:26.255,r,22.742,0.076
561,2025 MP24,2025-05-02 02:18:03.173,g,23.609,0.116


In [13]:
# Cross-match each observation in data_H with its band and rmsmag from obs_magnitudes.
# Same object (permid/provid) + same obstime identifies the observation.
keys = ['mpc_designation', 'obstime']
bands = obs_magnitudes[keys + ['band', 'rmsmag']].drop_duplicates(subset=keys)

data_H_band = data_H.merge(bands, on=keys, how='left', validate='many_to_one')
data_H_band['rmsmag'] = pd.to_numeric(data_H_band['rmsmag'], errors='coerce')  # stored as text
data_H_band

,obstime,mpc_row,mpc_designation,permid,provid,obs_et,dt_from_epoch_days,a,e,q,...,node,peri,M,r,delta,phase_angle,mag,H,band,rmsmag
0,2025-05-02 00:42:01.796,1487851,2025 MD8,None,2025 MD8,7.994186e+08,-402.970012,2.814204,0.119711,2.477313,...,115.86923,328.92434,98.353219,2.901265,2.039310,12.322578,23.096,18.498641,i,0.1240
1,2025-05-02 00:39:54.996,1487851,2025 MD8,None,2025 MD8,7.994185e+08,-402.971479,2.814204,0.119711,2.477313,...,115.86923,328.92434,98.352912,2.901264,2.039296,12.322145,22.927,18.329673,i,0.1270
2,2025-05-02 01:01:01.387,1487851,2025 MD8,None,2025 MD8,7.994197e+08,-402.956822,2.814204,0.119711,2.477313,...,115.86923,328.92434,98.355973,2.901280,2.039431,12.326470,23.055,18.457356,i,0.1050
3,2025-05-02 01:03:10.218,1487851,2025 MD8,None,2025 MD8,7.994199e+08,-402.955331,2.814204,0.119711,2.477313,...,115.86923,328.92434,98.356284,2.901282,2.039445,12.326910,23.059,18.461323,i,0.1290
4,2025-05-02 01:29:18.645,1487851,2025 MD8,None,2025 MD8,7.994214e+08,-402.937178,2.814204,0.119711,2.477313,...,115.86923,328.92434,98.360074,2.901303,2.039611,12.332264,22.911,18.312930,r,0.0720
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7954392,2025-07-18 01:03:03.946,1311281,2020 KJ56,None,2020 KJ56,8.060727e+08,-325.955404,43.840834,0.024141,42.782463,...,154.76027,143.11675,355.869914,42.785350,41.770096,0.061190,23.670,7.382559,Lr,0.1995
7954393,2025-08-17 07:07:02.087,1353570,2021 LD56,None,2021 LD56,8.086865e+08,-295.702647,40.233568,0.035963,38.786632,...,288.69354,17.80409,351.560323,38.803488,37.906157,0.700339,22.657,6.695863,Lr,0.1490
7954394,2025-08-17 07:03:07.476,1432311,2023 PJ5,None,2023 PJ5,8.086863e+08,-295.705363,47.487819,0.328316,31.896803,...,135.62001,160.71978,3.035565,31.945213,31.006109,0.688556,22.748,7.646804,Lr,0.1740
7954395,2025-07-08 08:36:06.169,1432312,2023 PK5,None,2023 PK5,8.052358e+08,-335.640795,42.116883,0.120041,37.061139,...,279.97838,65.56261,325.421686,38.182169,37.206504,0.434116,22.831,6.977308,Lz,0.2114


In [14]:
import numpy as np

def H_mean(H, rmsmag):
    errors = 1/rmsmag**2
    return np.sum(H * errors) / np.sum(errors)

def H_std(H, rmsmag):
    errors = 1/rmsmag**2
    return np.sqrt(np.sum(errors * (H - H_mean(H, rmsmag))**2) / np.sum(errors))

In [15]:
# Colors per object: weighted mean H in each band (H_mean), then differences of adjacent bands.
# Band labels come as 'Lr' and 'r' for the same filter, so drop the 'L' prefix.
data_H_band['band'] = data_H_band['band'].str.removeprefix('L')
obs = data_H_band.dropna(subset=['band', 'H', 'rmsmag']).copy()

# Same as H_mean(H, rmsmag) per object and band, vectorized (groupby.apply is slow on ~1M groups)
obs['w'] = 1 / obs['rmsmag']**2
obs['wH'] = obs['w'] * obs['H']
obs['wH2'] = obs['w'] * obs['H']**2
g = obs.groupby(['mpc_designation', 'band'])
sum_w = g['w'].sum()
mean = g['wH'].sum() / sum_w
# Weighted standard deviation: sqrt( sum(w * (H - mean)^2) / sum(w) )
std = np.sqrt((g['wH2'].sum() / sum_w - mean**2).clip(lower=0))
count = g.size()
H_stats = pd.DataFrame({'mean': mean,
                        'err': 1 / np.sqrt(sum_w),
                        'std': std.where(count > 1),  # undefined for a single observation
                        'count': count}).unstack('band')

bands_order = ['u', 'g', 'r', 'i', 'z', 'y']
H_band = H_stats['mean'].reindex(columns=bands_order).add_prefix('H_')
Herr_band = H_stats['err'].reindex(columns=bands_order).add_prefix('Herr_')
Hstd_band = H_stats['std'].reindex(columns=bands_order).add_prefix('Hstd_')
N_band = H_stats['count'].reindex(columns=bands_order).fillna(0).astype(int).add_prefix('N_')

# Identifiers and time span of the observations used for each object
ids = obs.groupby('mpc_designation').agg(permid=('permid', 'first'),
                                         provid=('provid', 'first'),
                                         obstime_first=('obstime', 'min'),
                                         obstime_last=('obstime', 'max'))

colors = ids.join(H_band).join(Herr_band).join(Hstd_band).join(N_band)
for b1, b2 in zip(bands_order[:-1], bands_order[1:]):
    colors[f'{b1}-{b2}'] = colors[f'H_{b1}'] - colors[f'H_{b2}']

colors = colors.reset_index()
colors

,mpc_designation,permid,provid,obstime_first,obstime_last,H_u,H_g,H_r,H_i,H_z,...,N_g,N_r,N_i,N_z,N_y,u-g,g-r,r-i,i-z,z-y
0,(100001) 1982 UC3,100001,1982 UC3,2025-07-04 07:46:47.693,2025-11-17 03:49:52.531,NaN,15.967276,15.513627,15.469449,15.473976,...,6,9,10,14,0,NaN,0.453649,0.044177,-0.004526,NaN
1,(100005) 1986 RY,100005,1986 RY,2025-06-22 05:28:05.490,2025-08-11 07:11:46.321,NaN,17.186704,16.224285,16.053308,16.084360,...,2,7,8,7,10,NaN,0.962419,0.170978,-0.031052,-0.040135
2,(100008) 1988 QZ,100008,1988 QZ,2025-07-11 08:03:20.299,2025-07-11 08:14:48.841,NaN,NaN,NaN,NaN,15.491862,...,0,0,0,3,0,NaN,NaN,NaN,NaN,NaN
3,(100009) 1988 RQ4,100009,1988 RQ4,2025-04-26 06:40:40.996,2025-07-23 05:30:12.745,18.597105,17.075226,16.521725,16.344700,16.412330,...,32,25,11,10,3,1.521879,0.553501,0.177025,-0.067630,-0.162957
4,(100014) 1989 SR4,100014,1989 SR4,2025-07-11 05:55:02.162,2025-07-23 08:09:31.662,NaN,NaN,15.220222,14.958835,15.085119,...,0,2,3,4,0,NaN,NaN,0.261387,-0.126285,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263380,2026 JZ11,None,2026 JZ11,2026-05-13 04:48:20.637,2026-07-07 03:14:10.026,NaN,NaN,7.908892,7.537540,7.385546,...,0,3,36,2,0,NaN,NaN,0.371352,0.151994,NaN
263381,2026 KA6,None,2026 KA6,2026-05-20 04:59:34.670,2026-07-12 01:23:23.490,NaN,NaN,7.250683,6.973625,6.734780,...,0,4,8,1,0,NaN,NaN,0.277057,0.238845,NaN
263382,2026 KB6,None,2026 KB6,2026-05-20 04:56:54.337,2026-07-07 03:15:30.490,NaN,NaN,7.046888,6.603348,6.732558,...,0,4,7,2,0,NaN,NaN,0.443540,-0.129209,NaN
263383,2026 KO6,None,2026 KO6,2026-05-20 04:47:35.740,2026-07-07 03:13:29.569,NaN,NaN,7.408188,6.907780,6.818608,...,0,1,10,1,0,NaN,NaN,0.500407,0.089172,NaN


In [16]:
colors.columns

Index(['mpc_designation', 'permid', 'provid', 'obstime_first', 'obstime_last',
       'H_u', 'H_g', 'H_r', 'H_i', 'H_z', 'H_y', 'Herr_u', 'Herr_g', 'Herr_r',
       'Herr_i', 'Herr_z', 'Herr_y', 'Hstd_u', 'Hstd_g', 'Hstd_r', 'Hstd_i',
       'Hstd_z', 'Hstd_y', 'N_u', 'N_g', 'N_r', 'N_i', 'N_z', 'N_y', 'u-g',
       'g-r', 'r-i', 'i-z', 'z-y'],
      dtype='object')

In [ ]:
colors.to_parquet('data/colors.parquet', engine='pyarrow', index=False)

: 

In [17]:
# Validation: compare our colors with the SDSS Moving Object Catalog (Ivezic et al., ADR4)
# https://faculty.washington.edu/ivezic/sdssmoc/sdssmoc.html
# SDSS values are the median colors over each asteroid's SDSS detections (errors < 0.03 mag).
sdss = pd.DataFrame({
    'mpc_designation': ['(1232) Cortusa', '(2405) Welch', '(4371) Fyodorov',
                        '(5302) Romanoserra', '(3573) Holmberg'],
    'g-r_sdss': [0.490, 0.470, 0.640, 0.735, 0.750],
    'r-i_sdss': [0.190, 0.120, 0.230, 0.280, 0.250],
    'i-z_sdss': [0.090, 0.110, -0.020, 0.135, -0.075],
})

check = sdss.merge(colors[['mpc_designation', 'N_g', 'N_r', 'N_i', 'N_z', 'g-r', 'r-i', 'i-z']],
                   on='mpc_designation', how='left')
for c in ['g-r', 'r-i', 'i-z']:
    check[f'{c}_diff'] = check[c] - check[f'{c}_sdss']

check[['mpc_designation', 'N_g', 'N_r', 'N_i', 'N_z',
       'g-r', 'g-r_sdss', 'g-r_diff',
       'r-i', 'r-i_sdss', 'r-i_diff',
       'i-z', 'i-z_sdss', 'i-z_diff']].round(3)

,mpc_designation,N_g,N_r,N_i,N_z,g-r,g-r_sdss,g-r_diff,r-i,r-i_sdss,r-i_diff,i-z,i-z_sdss,i-z_diff
0,(1232) Cortusa,24,92,41,33,0.449,0.490,-0.041,0.179,0.19,-0.011,0.144,0.090,0.054
1,(2405) Welch,7,7,9,9,0.295,0.470,-0.175,0.276,0.12,0.156,-0.018,0.110,-0.128
2,(4371) Fyodorov,5,15,13,9,0.598,0.640,-0.042,0.154,0.23,-0.076,-0.021,-0.020,-0.001
3,(5302) Romanoserra,5,10,10,10,0.733,0.735,-0.002,0.217,0.28,-0.063,0.120,0.135,-0.015
4,(3573) Holmberg,73,8,5,7,1.002,0.750,0.252,-0.210,0.25,-0.460,0.106,-0.075,0.181


In [18]:
import pandas as pd
colors_data = pd.read_parquet('data/colors.parquet', engine='pyarrow')

In [19]:
colors_data[colors_data['provid'] == '2025 MA45']

,mpc_designation,permid,provid,obstime_first,obstime_last,H_u,H_g,H_r,H_i,H_z,...,N_g,N_r,N_i,N_z,N_y,u-g,g-r,r-i,i-z,z-y
201842,2015 KR298,None,2025 MA45,2025-04-22 02:37:21.011,2025-05-04 03:40:31.048,20.053164,19.595598,19.204412,19.084263,NaN,...,51,59,44,0,0,0.457566,0.391186,0.120149,NaN,NaN
